In [2]:
import pandas as pd

hitter = pd.read_csv("../data/hitter_season_stats_2018_2026.csv")
pitcher = pd.read_csv("../data/pitcher_season_stats_2018_2026.csv")
fa = pd.read_csv("../data/fa_contracts.csv")

print("타자:", hitter.shape)
print("투수:", pitcher.shape)
print("FA:", fa.shape)

display(hitter.head())
display(fa.head())

타자: (1003, 27)
투수: (1452, 17)
FA: (89, 8)


,collect_year,playerId,playerName,teamName,isRetire,isPlayer,hitterGameCount,hitterAb,hitterHra,hitterObp,...,hitterH2,hitterH3,hitterBb,hitterKk,hitterSb,hitterCs,hitterGd,hitterWpa,hitterWar,sort_source
0,2018,78224,김재환,두산,N,Y,139,527,0.333966,0.405,...,36,1,59,134,2,NaN,NaN,3.839,9.39,hitterWar
1,2018,75125,박병호,넥센,Y,Y,113,400,0.345000,0.457,...,20,0,68,114,0,NaN,NaN,5.250,7.18,hitterWar
2,2018,67450,러프,삼성,N,Y,137,506,0.330040,0.419,...,32,4,65,107,5,NaN,NaN,4.102,7.12,hitterWar
3,2018,76267,최주환,두산,N,Y,138,519,0.333333,0.397,...,39,6,51,89,1,NaN,NaN,2.592,6.58,hitterWar
4,2018,79192,채은성,LG,N,Y,139,529,0.330813,0.379,...,36,2,35,97,3,NaN,NaN,2.205,6.52,hitterWar


,player_name,fa_year,age_at_fa,position,contract_years,total_contract_amount,annual_avg_salary,team
0,이용찬,2021,NaN,P,4,27.0,6.75,NC
1,유희관,2021,NaN,P,1,10.0,10.00,두산
2,차우찬,2021,NaN,P,2,20.0,10.00,LG
3,이대호,2021,NaN,1B,2,26.0,13.00,롯데
4,김상수,2021,NaN,P,3,15.5,5.17,SK


In [3]:
fa["position_group"] = fa["position"].apply(lambda x: "pitcher" if x == "P" else "hitter")

print(fa["position_group"].value_counts())
display(fa.head())

position_group
hitter     55
pitcher    34
Name: count, dtype: int64


,player_name,fa_year,age_at_fa,position,contract_years,total_contract_amount,annual_avg_salary,team,position_group
0,이용찬,2021,NaN,P,4,27.0,6.75,NC,pitcher
1,유희관,2021,NaN,P,1,10.0,10.00,두산,pitcher
2,차우찬,2021,NaN,P,2,20.0,10.00,LG,pitcher
3,이대호,2021,NaN,1B,2,26.0,13.00,롯데,hitter
4,김상수,2021,NaN,P,3,15.5,5.17,SK,pitcher


In [4]:
# FA 연도 기준 직전 3개년 데이터가 실제로 존재하는지 확인

def check_hitter_rows(row):
    name = row["player_name"]
    fa_year = row["fa_year"]
    years = [fa_year - 3, fa_year - 2, fa_year - 1]

    matched = hitter[
        (hitter["playerName"] == name) &
        (hitter["collect_year"].isin(years))
    ]

    return len(matched)

def check_pitcher_rows(row):
    name = row["player_name"]
    fa_year = row["fa_year"]
    years = [fa_year - 3, fa_year - 2, fa_year - 1]

    matched = pitcher[
        (pitcher["playerName"] == name) &
        (pitcher["collect_year"].isin(years))
    ]

    return len(matched)

fa["matched_year_count"] = fa.apply(
    lambda row: check_pitcher_rows(row) if row["position_group"] == "pitcher" else check_hitter_rows(row),
    axis=1
)

print(fa["matched_year_count"].value_counts().sort_index())
display(fa[["player_name", "fa_year", "position", "position_group", "matched_year_count"]].head(30))

matched_year_count
0     4
1     8
2    14
3    63
Name: count, dtype: int64


,player_name,fa_year,position,position_group,matched_year_count
0,이용찬,2021,P,pitcher,2
1,유희관,2021,P,pitcher,3
2,차우찬,2021,P,pitcher,2
3,이대호,2021,1B,hitter,3
4,김상수,2021,P,pitcher,0
5,김재호,2021,SS,hitter,3
6,우규민,2021,P,pitcher,3
7,이원석,2021,3B,hitter,3
8,정수빈,2021,OF,hitter,3
9,최형우,2021,OF,hitter,3


In [5]:
import pandas as pd
import numpy as np

# 타자만 분리
fa_hitter = fa[fa["position_group"] == "hitter"].copy()

training_rows = []

for _, row in fa_hitter.iterrows():

    player = row["player_name"]
    fa_year = row["fa_year"]

    target_years = [
        fa_year - 3,
        fa_year - 2,
        fa_year - 1
    ]

    stats = hitter[
        (hitter["playerName"] == player) &
        (hitter["collect_year"].isin(target_years))
    ]

    if len(stats) < 2:
        continue

    training_rows.append({

        "player_name": player,
        "fa_year": fa_year,

        "war_3yr_avg": stats["hitterWar"].mean(),
        "ops_3yr_avg": stats["hitterOps"].mean(),
        "wrc_plus_3yr_avg": stats["hitterWrcPlus"].mean(),
        "games_3yr_avg": stats["hitterGameCount"].mean(),

        "annual_avg_salary": row["annual_avg_salary"]

    })

hitter_training = pd.DataFrame(training_rows)

print(hitter_training.shape)

display(hitter_training.head())

(47, 7)


,player_name,fa_year,war_3yr_avg,ops_3yr_avg,wrc_plus_3yr_avg,games_3yr_avg,annual_avg_salary
0,이대호,2021,3.556667,0.861000,107.466667,141.000000,13.00
1,김재호,2021,2.630000,0.771667,121.000000,127.000000,8.33
2,이원석,2021,1.630000,0.794333,90.833333,120.000000,6.67
3,정수빈,2021,2.260000,0.782333,123.966667,96.666667,9.33
4,최형우,2021,5.796667,0.961333,149.966667,139.666667,15.67


In [7]:
import pandas as pd
import numpy as np

def convert_inning(value):
    """
    네이버 이닝 문자열을 숫자로 변환.
    예:
    '145 2/3' -> 145.666...
    '62 1/3' -> 62.333...
    '180' -> 180.0
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if " " in value:
        whole, fraction = value.split(" ")
        numerator, denominator = fraction.split("/")
        return float(whole) + float(numerator) / float(denominator)

    if "/" in value:
        numerator, denominator = value.split("/")
        return float(numerator) / float(denominator)

    return float(value)


numeric_cols = [
    "pitcherWar",
    "pitcherEra",
    "pitcherWhip",
    "pitcherInning",
    "pitcherGameCount"
]

for col in numeric_cols:
    if col == "pitcherInning":
        pitcher[col] = pitcher[col].apply(convert_inning)
    else:
        pitcher[col] = pd.to_numeric(pitcher[col], errors="coerce")

print(pitcher[numeric_cols].dtypes)
display(pitcher[numeric_cols].head())

pitcherWar          float64
pitcherEra          float64
pitcherWhip         float64
pitcherInning       float64
pitcherGameCount      int64
dtype: object


,pitcherWar,pitcherEra,pitcherWhip,pitcherInning,pitcherGameCount
0,5.66,3.84422,1.20,199.000000,31
1,5.28,2.88142,1.07,168.666667,26
2,4.82,4.25237,1.41,175.666667,29
3,4.59,2.97794,1.14,136.000000,25
4,4.54,3.07059,1.14,170.000000,26


In [8]:
fa_pitcher = fa[fa["position_group"] == "pitcher"].copy()

training_rows = []

for _, row in fa_pitcher.iterrows():

    player = row["player_name"]
    fa_year = row["fa_year"]

    target_years = [
        fa_year - 3,
        fa_year - 2,
        fa_year - 1
    ]

    stats = pitcher[
        (pitcher["playerName"] == player) &
        (pitcher["collect_year"].isin(target_years))
    ]

    if len(stats) < 2:
        continue

    training_rows.append({
        "player_name": player,
        "fa_year": fa_year,

        "war_3yr_avg": stats["pitcherWar"].mean(),
        "era_3yr_avg": stats["pitcherEra"].mean(),
        "innings_3yr_avg": stats["pitcherInning"].mean(),
        "games_3yr_avg": stats["pitcherGameCount"].mean(),

        "annual_avg_salary": row["annual_avg_salary"]
    })

pitcher_training = pd.DataFrame(training_rows)

print(pitcher_training.shape)
display(pitcher_training.head())

(30, 7)


,player_name,fa_year,war_3yr_avg,era_3yr_avg,innings_3yr_avg,games_3yr_avg,annual_avg_salary
0,이용찬,2021,2.205,3.847585,146.166667,25.500000,6.75
1,유희관,2021,0.710,4.988577,147.888889,28.000000,10.00
2,차우찬,2021,-0.080,5.102535,169.166667,29.000000,10.00
3,우규민,2021,1.090,4.409570,55.222222,51.333333,5.00
4,백정현,2022,3.275,3.433915,157.333333,27.500000,9.50


In [9]:
hitter_training.to_csv(
    "../data/hitter_training.csv",
    index=False,
    encoding="utf-8-sig"
)

pitcher_training.to_csv(
    "../data/pitcher_training.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("hitter_training:", hitter_training.shape)
print("pitcher_training:", pitcher_training.shape)

저장 완료
hitter_training: (47, 7)
pitcher_training: (30, 7)
